# AdaptiveSRE Colab Validation Notebook

This notebook is set up for Google Colab validation.

Flow:
1. Install dependencies
2. Clone the repo and start mock services as Python processes
3. Start the main server and verify `/health`
4. Run a short training smoke test
5. Run full training
6. Evaluate the trained checkpoint
7. Generate the reward plot

Success criteria:
- `train.py` completes 10 episodes without crashing
- `positive_trajectories > 0`
- `train.py` completes 200 episodes
- `eval.py` shows Gen1 > Gen0
- `rewards_curve.png` is generated

If Colab fails, the likely culprits are:
- `CUDA out of memory`: reduce `batch_size` in `train.py`
- `unsloth` import error: re-run the install cell
- Mock services do not start: check port conflicts
- `Connection refused` to `localhost:8000`: server did not start
- All rewards are `0.001`: debug `grader.py`
- Model outputs invalid JSON: tune temperature or prompt formatting in `train.py`

In [ ]:
# Cell 1: Install all AdaptiveSRE deps with pinned versions
import subprocess

# Upgrade pip first
subprocess.run(["pip", "install", "-q", "--upgrade", "pip", "virtualenv"], check=True)
subprocess.run(["python3", "-m", "virtualenv", "/content/adaptive_venv"], check=True)

PIP = "/content/adaptive_venv/bin/pip"

# Core ML stack — pinned versions
packages = [
    "torch==2.4.0",
    "transformers==4.44.0",
    "trl==0.9.6",          # MUST match train.py — do not upgrade
    "accelerate==0.33.0",
    "peft==0.12.0",
    "bitsandbytes==0.43.3",
    "datasets==2.21.0",
    # Server stack
    "fastapi==0.115.0",
    "uvicorn==0.30.0",
    "pydantic==2.7.0",
    "httpx==0.27.0",
    "gradio==4.44.0",
    "huggingface_hub",
    "matplotlib",
    "protobuf<4",
]

subprocess.run([PIP, "install", "-q"] + packages, check=True)

# Unsloth — must come after torch, install from source for Colab CUDA compat
subprocess.run([
    PIP, "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)

# Remove wandb to avoid proto conflicts
subprocess.run([PIP, "uninstall", "-y", "wandb"], capture_output=True)

# Verify
subprocess.run([
    "/content/adaptive_venv/bin/python", "-c",
    "import torch, trl, unsloth; print('ALL IMPORTS OK'); "
    "print('torch:', torch.__version__); "
    "print('trl:', trl.__version__); "
    "print('CUDA:', torch.cuda.is_available())"
], check=True)

In [ ]:
# Cell 2: Clone repo, fix ports, start services, verify GPU

!if [ ! -d Adaptive-SRE ]; then git clone https://github.com/ashifsekh/Adaptive-SRE.git; fi
%cd Adaptive-SRE

PY = "/content/adaptive_venv/bin/python"

# GPU check FIRST — stop here if no GPU
!/content/adaptive_venv/bin/python -c "
import torch
assert torch.cuda.is_available(), 'NO GPU! Go to Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {round(torch.cuda.get_device_properties(0).total_memory/1e9,1)} GB')
"

# Start mock services — use safe ports (no conflicts)
import subprocess, time, urllib.request

services = [
    ("mock_services.db.main:app",           15432),  # 5432 blocked
    ("mock_services.auth.main:app",          8102),
    ("mock_services.payment.main:app",       8101),
    ("mock_services.cache.main:app",        16379),  # 6379 = Redis conflict
    ("mock_services.notification.main:app",  8103),
]

for module, port in services:
    subprocess.Popen(
        ["/content/adaptive_venv/bin/python", "-m", "uvicorn",
         module, "--host", "0.0.0.0", "--port", str(port)],
        stdout=open(f"{module.split('.')[2]}.log", "w"),
        stderr=subprocess.STDOUT
    )

# Start main server
subprocess.Popen(
    ["/content/adaptive_venv/bin/python", "-m", "uvicorn",
     "server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT
)

# Wait for health
for attempt in range(40):
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            print("Server healthy:", r.read().decode())
            break
    except:
        time.sleep(1)
else:
    print("SERVER FAILED. Last 30 lines of server.log:")
    print(open("server.log").read()[-2000:])
    raise SystemExit(1)

In [ ]:
# Cell 2.5: Sanity check — confirm full pipeline works before training
import httpx, json

with httpx.Client(timeout=10) as c:
    # Reset easy task
    r = c.post("http://localhost:8000/reset", json={"task": "easy"})
    obs = r.json()
    print("Reset OK:", obs.get("episode_id"))

    # Take one step
    action = {
        "command": "docker stats --no-stream",
        "reasoning": "test probe",
        "approach": "probe",
        "drift_detected": False,
        "lead_mode_guess": "unknown",
        "root_cause_guess": None
    }
    r2 = c.post("http://localhost:8000/step", json=action)
    result = r2.json()
    reward = result["reward"]
    print(f"Step OK: reward={reward:.4f}, done={result['done']}")
    assert 0.001 <= reward <= 0.999, f"REWARD OUT OF RANGE: {reward}"
    print("✅ All sanity checks passed — safe to start training")

In [ ]:
# Cell 3: Short training smoke test (10 episodes, 5-10 mins on T4)
!/content/adaptive_venv/bin/python train.py --episodes 10 --task easy --output ./test_checkpoints/ --env_url http://localhost:8000

In [ ]:
# Cell 4: If step 3 works, run real training (200 episodes, ~1-2 hours)
!/content/adaptive_venv/bin/python train.py --episodes 200 --task hard --output ./checkpoints/gen1/ --env_url http://localhost:8000

In [ ]:
# Cell 5: Evaluate
!/content/adaptive_venv/bin/python eval.py --trained_model ./checkpoints/gen1/final --env_url http://localhost:8000 --output eval_results.json

# Cell 6: Plot
!/content/adaptive_venv/bin/python plot_rewards.py
from IPython.display import Image
Image('rewards_curve.png')